# Package

In [30]:
import numpy as np
from pathlib import Path
import pandas as pd
from dateutil.relativedelta import relativedelta

from feast import FeatureStore
import pickle

from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importation des données

In [17]:
# ----------------------------
# 0) Config
# ----------------------------
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO","M2SL","OILPRICEX",
    "RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

# Feast repo (tu lances depuis feature_repo/, donc "." marche aussi)
REPO_PATH = (
    Path.cwd().parent
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

store = FeatureStore(repo_path=str(REPO_PATH))
print("Project:", store.project)
print("Feature views:", [fv.name for fv in store.list_feature_views()])

Project: unemployment_feature_store
Feature views: ['stationary_value', 'raw_value']


In [18]:
# ----------------------------
# 1) Construire les dates (SANS DATES_PATH)
#    -> on génère un calendrier puis Feast filtrera ce qui existe réellement
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

# Pour récupérer une liste de dates "validées" par Feast :
# on interroge UNRATE seulement, puis on prend les dates retournées.
entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})

df_unrate = store.get_historical_features(
    entity_df=entity_df_unrate,
    features=["raw_value:value"],          # juste pour valider les dates existantes
    full_feature_names=True,
).to_df()

# Normalisation date + tri + unique
dates = (
    pd.to_datetime(df_unrate["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)                 # enlève le +00:00
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00


In [19]:
# ----------------------------
# 2) Entity DF multi-séries
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

print("entity_df shape:", entity_df.shape)
print(entity_df.head())

entity_df shape: (8811, 2)
  series_id       date
0  BUSLOANS 1959-01-01
1  BUSLOANS 1959-02-01
2  BUSLOANS 1959-03-01
3  BUSLOANS 1959-04-01
4  BUSLOANS 1959-05-01


In [20]:
# ----------------------------
# 3) Fetch stationary features (Feast)
# ----------------------------
df_stationary = store.get_historical_features(
    entity_df=entity_df,
    features=["stationary_value:value"],
    full_feature_names=True,               # IMPORTANT
).to_df()

# Normaliser types
df_stationary["date"] = pd.to_datetime(df_stationary["date"], utc=True, errors="coerce").dt.tz_convert(None)

# Colonne valeur robuste (au cas où)
value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    # fallback (rare)
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Colonne attendue '{value_col}' absente. Trouvé: {candidates}")

print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date  stationary_value__value
0  BUSLOANS 1960-01-01                 0.011578
1    INDPRO 1960-01-01                 0.091976
2     USREC 1960-01-01                 0.000000
3      M2SL 1960-01-01                 0.001323
4  CPIAUCSL 1960-01-01                -0.006156


In [21]:
# ----------------------------
# 4) Pivot LONG → WIDE (1 colonne par série)
# ----------------------------
df = (
    df_stationary
    .pivot_table(index="date", columns="series_id", values=value_col, aggfunc="last")
    .sort_index()
)

# Borner la date max à fin août 2025 (dernier mois complet)
df = df.loc[:pd.Timestamp("2025-08-01")]

# (Optionnel) réordonner les colonnes selon series_ids
df = df.reindex(columns=series_ids)

print("df shape:", df.shape)
df

df shape: (788, 11)


series_id,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,UNRATE,USREC
date,,,,,,,,,,,
1960-01-01,0.011578,-0.006156,0.001204,0.091976,0.001323,0.000000,0.020977,0.017909,0.30,-0.8,0.0
1960-02-01,0.011905,-0.003767,0.006009,0.076960,0.002007,0.000000,0.014565,-0.025663,-0.19,-1.1,0.0
1960-03-01,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.000000,0.006250,-0.070857,-1.18,-0.2,0.0
1960-04-01,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.000000,0.006489,-0.040442,-1.12,0.0,0.0
1960-05-01,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.000000,0.007747,-0.010090,-0.67,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-04-01,0.046085,-0.007236,0.007959,0.011685,0.004244,-0.226416,0.015108,-0.107606,0.00,0.3,0.0
2025-05-01,0.044487,-0.007941,0.007758,0.000360,0.004820,-0.162581,0.008190,-0.038448,0.03,0.2,0.0
2025-06-01,0.035593,-0.000435,0.002578,0.005179,0.004174,0.026151,0.001403,0.059087,0.03,0.0,0.0


# Run_config

In [22]:
# ---------- Paramètres généraux ----------
h = 12
min_train_n = 36           # ≥ 3 ans avant de commencer à prévoir
winsor_level = 0.01        # winsorisation (1er/99e percentiles)
norm_var = True            # normaliser ou non
target_col = "UNRATE"      # cible dans df_stationary

# Fenêtres d'évaluation / test
eval_start = pd.Timestamp("1983-01-01")
eval_end   = pd.Timestamp("1989-12-31")
test_start = pd.Timestamp("1990-01-01")
test_end   = pd.Timestamp("2025-12-31")   # ajuste si besoin

# ---------- Bagging (bootstrap en blocs) ----------
use_bagging = True
B_boot = 30               # comme les auteurs
L_block = 12              # blocs annuels (12 mois)
rng = np.random.default_rng(123)  # seed bootstrap

# ---------- Paramètres Conformal ----------
PI_WINDOWS = 24      # nb de fenêtres passées
STEP_SIZE  = 12      # espacement entre cutoffs (en mois)
ALPHA      = 0.05    # 95%

# ---------- Fichiers de sortie ----------
LINREG_PKL  = "linear_regression.pkl"        # bundle (dict)
LINREG_META = "linear_regression_meta.csv"   # méta résumé

In [23]:
# ---------- Préparation df_stationary ----------
def _ensure_ms_index(df):
    """Force un index DatetimeIndex en début de mois (MS)."""
    if "date" in df.columns:
        df = df.set_index("date")
    idx = pd.to_datetime(df.index)
    df = df.copy()
    df.index = idx.to_period("M").to_timestamp(how="start")
    return df.asfreq("MS")

# On part de df_stationary (toutes données : 1960→2025), déjà chargé en mémoire
df_all = _ensure_ms_index(df).sort_index()

if target_col not in df_all.columns:
    raise ValueError(f"La colonne cible '{target_col}' est absente de df_stationary.")

y_all = df_all[target_col].astype(float)
X_all = df_all.drop(columns=[target_col]).astype(float)
features = list(X_all.columns)

print(f"✅ Données prêtes : {df_all.index.min().date()} → {df_all.index.max().date()} | n={len(df_all)} | freq=MS")
print(f"Features ({len(features)}): {features[:6]}{' ...' if len(features)>6 else ''}")

✅ Données prêtes : 1960-01-01 → 2025-08-01 | n=788 | freq=MS
Features (10): ['BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX'] ...


In [24]:
# ---------- Préproc ----------
def fit_preproc(X, wins=0.01, do_norm=True):
    """Apprend winsor + normalisation sur TRAIN et renvoie (X_trans, prep)."""
    lower = X.quantile(wins)
    upper = X.quantile(1 - wins)
    Xw = X.clip(lower=lower, upper=upper, axis=1)
    if do_norm:
        mean = Xw.mean()
        std  = Xw.std().replace(0, 1)
        Xn   = (Xw - mean) / std
        prep = {"lower": lower, "upper": upper, "mean": mean, "std": std, "norm": True}
        return Xn, prep
    else:
        prep = {"lower": lower, "upper": upper, "mean": None, "std": None, "norm": False}
        return Xw, prep

def apply_preproc(X, prep):
    """Applique le préproc appris (pas de fuite)."""
    Xp = X.clip(lower=prep["lower"], upper=prep["upper"], axis=1)
    if prep["norm"]:
        Xp = (Xp - prep["mean"]) / prep["std"].replace(0, 1)
    return Xp

In [25]:
# ---------- Bootstrap utils ----------
def block_bootstrap_rows(index, L, rng):
    """
    Moving-block bootstrap sur index (positions).
    Renvoie un array d'indices (longueur = n).
    """
    n = len(index)
    if n < 3:
        return np.arange(n)  # fallback
    L = max(2, min(int(L), n-1))
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    ix = np.concatenate([np.arange(s, s+L) for s in starts])[:n]
    return ix

def bagged_predict_linreg(X_tr_raw, y_tr, x_fore_raw, prep, B, L, rng):
    """
    Bagging (moving-block bootstrap) pour LinearRegression :
      - préproc fixé sur TRAIN original (pas ré-appris)
      - rééchantillon par blocs (lignes) (X, y)
      - fit et prédiction h
      - renvoie (moyenne, distribution complète, base_pred)
    """
    # Base fit (référence)
    X_tr_p = apply_preproc(X_tr_raw, prep)
    base = LinearRegression()
    base.fit(X_tr_p, y_tr.values)
    yhat_base = float(base.predict(apply_preproc(x_fore_raw, prep))[0])

    preds = []
    for _ in range(B):
        ix = block_bootstrap_rows(X_tr_raw.index, L, rng)
        Xb = X_tr_raw.iloc[ix]
        yb = y_tr.iloc[ix]
        Xb_p = apply_preproc(Xb, prep)  # IMPORTANT: même prep
        m = LinearRegression()
        m.fit(Xb_p, yb.values)
        preds.append(float(m.predict(apply_preproc(x_fore_raw, prep))[0]))
    return float(np.mean(preds)), np.array(preds), yhat_base

In [26]:
import numpy as np
from sklearn.linear_model import LinearRegression

def fit_predict_lr_base(X_tr_raw, y_tr, x_fore_raw, prep):
    """Fit LR (base) sur train, prédit sur x_fore (1 ligne)."""
    m = LinearRegression()
    m.fit(apply_preproc(X_tr_raw, prep), y_tr.values)
    return float(m.predict(apply_preproc(x_fore_raw, prep))[0])


def conformal_q_from_past_windows_lr(
    y_all, X_all, t_end, *,
    h=12,
    step_size=12,
    pi_windows=24,
    alpha=0.05,
    min_train_n=36,
    winsor_level=0.01,
    norm_var=True,
):
    """
    Fenêtres de calibration (comme ton ARp) :
      t_cal = t_end - k*step_size
      compare y[t_cal+h] vs yhat_base(train jusqu'à t_cal)
    q = quantile(1-alpha) des |erreurs| avec correction finie.
    """
    errs = []

    for k in range(1, pi_windows + 1):
        t_cal = t_end - relativedelta(months=k * step_size)
        t_cal_fore = t_cal + relativedelta(months=h)

        # doit exister dans l'index
        if (t_cal not in y_all.index) or (t_cal_fore not in y_all.index):
            continue

        # train jusqu'à t_cal
        y_tr = y_all.loc[:t_cal].dropna()
        X_tr = X_all.loc[:t_cal]

        # align et dropna sur features
        X_tr = X_tr.loc[y_tr.index].dropna()
        y_tr = y_tr.loc[X_tr.index]

        if len(y_tr) < min_train_n:
            continue

        # future exog à t_cal_fore
        try:
            x_fore_raw = X_all.loc[[t_cal_fore]]
        except Exception:
            continue
        if x_fore_raw.isna().any(axis=None):
            continue

        # préproc appris sur train de la fenêtre
        try:
            _, prep = fit_preproc(X_tr, wins=winsor_level, do_norm=norm_var)
            yhat_cal = fit_predict_lr_base(X_tr, y_tr, x_fore_raw, prep)
        except Exception:
            continue

        y_true = float(y_all.loc[t_cal_fore])
        err = abs(y_true - yhat_cal)
        if np.isfinite(err):
            errs.append(err)

    if len(errs) == 0:
        return np.nan

    # quantile conformal (finite-sample correction)
    n = len(errs)
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(max(q_level, 0.0), 1.0)

    return float(np.quantile(errs, q_level))

In [27]:
# ---------- Boucle pseudo-OOS ----------
rows = []  # (date, y_pred, y_true, y_pred_base, p05, p95, lo95, hi95)
models = []
preprocs = []
train_ends = []

last_t_end = y_all.index.max() - relativedelta(months=h)
last_model = None
last_fit_end = None

for t_end in y_all.index:
    if t_end > last_t_end:
        break

    y_tr = y_all.loc[:t_end]
    X_tr = X_all.loc[:t_end]

    # (option) align + dropna pour éviter mismatch / NaN features
    y_tr = y_tr.dropna()
    X_tr = X_tr.loc[y_tr.index].dropna()
    y_tr = y_tr.loc[X_tr.index]

    if len(y_tr) < min_train_n:
        continue

    # Préproc appris sur TRAIN courant
    X_tr_p, prep = fit_preproc(X_tr, wins=winsor_level, do_norm=norm_var)

    # Horizon ciblé
    t_fore = t_end + relativedelta(months=h)
    if t_fore in y_all.index:
        x_fore_raw = X_all.loc[[t_fore]]

        # skip si exog future manquante
        if x_fore_raw.isna().any(axis=None):
            continue

        # 1) prédiction (bagging ou base)
        if use_bagging:
            yhat_h, dist, yhat_base = bagged_predict_linreg(
                X_tr_raw=X_tr, y_tr=y_tr, x_fore_raw=x_fore_raw,
                prep=prep, B=B_boot, L=L_block, rng=rng
            )
            y_p05 = float(np.percentile(dist, 5))
            y_p95 = float(np.percentile(dist, 95))
        else:
            model = LinearRegression()
            model.fit(X_tr_p, y_tr.values)
            yhat_h = float(model.predict(apply_preproc(x_fore_raw, prep))[0])
            yhat_base = yhat_h
            y_p05, y_p95 = (np.nan, np.nan)

        # 2) Conformal q (past windows) -> intervalle autour de yhat_h
        q = conformal_q_from_past_windows_lr(
            y_all=y_all,
            X_all=X_all,
            t_end=t_end,
            h=h,
            step_size=STEP_SIZE,
            pi_windows=PI_WINDOWS,
            alpha=ALPHA,
            min_train_n=min_train_n,
            winsor_level=winsor_level,
            norm_var=norm_var,
        )

        if np.isfinite(q):
            y_lo95 = float(yhat_h - q)
            y_hi95 = float(yhat_h + q)
        else:
            y_lo95, y_hi95 = (np.nan, np.nan)

        rows.append((
            t_fore,
            float(yhat_h),
            float(y_all.loc[t_fore]),
            float(yhat_base),
            float(y_p05) if np.isfinite(y_p05) else np.nan,
            float(y_p95) if np.isfinite(y_p95) else np.nan,
            y_lo95,
            y_hi95
        ))

    # trace / dernier modèle base (utile pour sauvegarde)
    last_model = LinearRegression().fit(X_tr_p, y_tr.values)
    last_fit_end = t_end
    models.append(last_model)
    preprocs.append(prep)
    train_ends.append(t_end)

In [28]:
# ---------- DataFrame OOS ----------
if rows:
    df_oos = (
        pd.DataFrame(
            rows,
            columns=[
                "date",
                "y_pred",          # prédiction baggée
                "y_true",
                "y_pred_base",     # prédiction modèle base
                "y_pred_p05",      # bagging p05 (optionnel)
                "y_pred_p95",      # bagging p95 (optionnel)
                "y_lo_95",         # conformal (base)
                "y_hi_95",         # conformal (base)
            ],
        )
        .assign(date=lambda d: pd.to_datetime(d["date"])
                            .dt.to_period("M")
                            .dt.to_timestamp(how="start"))
        .set_index("date")
        .sort_index()
    )
else:
    df_oos = pd.DataFrame(
        columns=[
            "y_pred", "y_true", "y_pred_base",
            "y_pred_p05", "y_pred_p95",
            "y_lo_95", "y_hi_95",
        ]
    )
    df_oos.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos)}")
print(df_oos.head(3))


✅ Pseudo-OOS terminé — n prévisions = 741
              y_pred  y_true  y_pred_base  y_pred_p05  y_pred_p95  y_lo_95  \
date                                                                         
1963-12-01 -0.716835     0.0    -0.288306   -1.181973   -0.177450      NaN   
1964-01-01 -0.244223    -0.1    -0.180196   -1.125814    0.670957      NaN   
1964-02-01  1.109324    -0.5     1.288267   -0.095926    2.216658      NaN   

            y_hi_95  
date                 
1963-12-01      NaN  
1964-01-01      NaN  
1964-02-01      NaN  


In [31]:
# ---------- Sauvegardes ----------
bundle = {
    "oos_predictions": df_oos.reset_index(),     # (date, y_pred, y_true, y_pred_base, y_pred_p05, y_pred_p95)
    "params": {
        "model": "LinearRegression",
        "horizon": h,
        "min_train_n": min_train_n,
        "winsor_level": winsor_level,
        "norm_var": norm_var,
        "features": features,
        "eval_window": (str(eval_start.date()), str(eval_end.date())),
        "test_window": (str(test_start.date()), str(test_end.date())),
        # ---- bagging ----
        "use_bagging": bool(use_bagging),
        "B_boot": int(B_boot),
        "L_block": int(L_block),
    },
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_all": int(len(df_all)),
        "n_forecasts": int(len(df_oos)),
    },
    "train_fit_dates": pd.to_datetime(pd.Index(train_ends)),

    # 🔻🔻🔻 AJOUT ESSENTIEL POUR LA PERMUTATION 🔻🔻🔻
    "models":   models,     # liste des modèles LinearRegression (un par fenêtre)
    "preprocs": preprocs,   # liste des préproc (dict) alignés aux modèles
    # 🔺🔺🔺
}

# --- Sauvegarde du bundle complet ---
with open(LINREG_PKL, "wb") as f:
    pickle.dump(bundle, f)

# --- Sauvegarde du résumé méta séparé (lisible rapidement) ---
pd.DataFrame([{
    "model": "LinearRegression",
    "horizon": h,
    "min_train_n": min_train_n,
    "winsor_level": winsor_level,
    "norm_var": norm_var,
    "use_bagging": bool(use_bagging),
    "B_boot": int(B_boot),
    "L_block": int(L_block),
    "trained_until": bundle["meta"]["trained_until"],
    "n_forecasts": bundle["meta"]["n_forecasts"],
}]).to_csv(LINREG_META, index=False)

print(f"\n💾 Bundle sauvegardé → {LINREG_PKL}")
print(f"💾 Méta sauvegardée → {LINREG_META}")
print(f"📦 Contenu du bundle : {list(bundle.keys())}")


💾 Bundle sauvegardé → linear_regression.pkl
💾 Méta sauvegardée → linear_regression_meta.csv
📦 Contenu du bundle : ['oos_predictions', 'params', 'meta', 'train_fit_dates', 'models', 'preprocs']


# Graphique

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from utilsforecast.plotting import plot_series

# =========================
# cellule 7 : Préparer df_obs + df_fcst (LR) (format utilsforecast)
# =========================
SERIES_ID = "UNRATE"

# df_obs : observations
df_obs = (
    df_oos
    .reset_index()
    .rename(columns={"date": "ds", "y_true": "y"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "y"]]
)

# df_fcst : forecasts + intervalles conformal
df_fcst = (
    df_oos
    .reset_index()
    .rename(columns={
        "date": "ds",
        "y_pred": "LR",
        "y_lo_95": "LR-lo-95",
        "y_hi_95": "LR-hi-95",
    })
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "LR", "LR-lo-95", "LR-hi-95"]]
)

print(df_obs.head(2))
print(df_fcst.head(2))

# =========================
# cellule 9 : Zoom + cadrage de l’axe Y
# =========================
START_ZOOM = "1990-01-01"
END_ZOOM   = "2025-08-01"

segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-fin"),
]

# Sécurité datetime
df_obs["ds"]  = pd.to_datetime(df_obs["ds"])
df_fcst["ds"] = pd.to_datetime(df_fcst["ds"])

start_zoom_dt = pd.to_datetime(START_ZOOM)
end_zoom_dt   = pd.to_datetime(END_ZOOM)

df_obs_z  = df_obs.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")
df_fcst_z = df_fcst.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")

fig = plot_series(
    df=df_obs_z,
    forecasts_df=df_fcst_z,
    level=[95],
    engine="plotly",
).update_layout(
    height=450,
    # (option) fixe l'axe Y si tu veux un cadrage constant
    # yaxis=dict(range=[-10, 12]),
    # enlève les carreaux
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
)

# =========================
# Renommage de la légende
# =========================
for trace in fig.data:
    name = (trace.name or "").lower()
    if trace.name == "y":
        trace.name = "Observed"
    elif trace.name == "LR":
        trace.name = "Linear Regression forecast"
    elif "95" in name or "lo" in name or "hi" in name:
        trace.name = "Conformal Prediction"

# =========================
# Un seul bouton "Segments" (sans 1990)
# =========================
# ymin/ymax auto à partir des données
ymin = min(df_obs_z["y"].min(), df_fcst_z["LR-lo-95"].min())
ymax = max(df_obs_z["y"].max(), df_fcst_z["LR-hi-95"].max())

SEG_GROUP = "SEGMENTS"
fig.update_layout(legend=dict(groupclick="togglegroup"))

first = True
for i, (start, _, _) in enumerate(segments):

    if i == 0:
        continue  # ignore 1990

    x = pd.to_datetime(start)
    if not (start_zoom_dt <= x <= end_zoom_dt):
        continue

    fig.add_trace(
        go.Scatter(
            x=[x, x],
            y=[ymin, ymax],
            mode="lines",
            legendgroup=SEG_GROUP,
            name="Segments" if first else None,
            showlegend=first,
            visible="legendonly",
            line=dict(color="gray", width=1, dash="dash"),
            hoverinfo="skip",
        )
    )
    first = False

fig.show()

  unique_id         ds    y
0    UNRATE 1963-12-01  0.0
1    UNRATE 1964-01-01 -0.1
  unique_id         ds        LR  LR-lo-95  LR-hi-95
0    UNRATE 1963-12-01 -0.716835       NaN       NaN
1    UNRATE 1964-01-01 -0.244223       NaN       NaN
